### Purpose

Lambda `dsecountProducts` 를 배포하고 API Gateway `jyipsj28s9` 에 `GET /ecountproducts` 로 연결한다.

- 소스: `dsecountProducts/lambda_function.py` (s3://dsbaseinfo/ecount/products_latest.json 을 읽어 반환)
- 구성은 기존 `/dschemical` 과 동일: 인증 없음, 패스스루 매핑 템플릿, CORS(`*`), 역할 `DSmapLambda`
- 여러 번 실행해도 된다 (있으면 갱신)
- **마지막 셀(`dev` 스테이지 배포)은 API 전체를 다시 배포한다** — 콘솔에서 배포 안 한 변경이 있으면 같이 반영됨

In [ ]:
# Norton 등이 HTTPS를 가로채는 PC에서 SSL 오류 방지 - boto3 보다 먼저 적용해야 함
import truststore
truststore.inject_into_ssl()

import io
import json
import zipfile
import urllib.request

import boto3
from botocore.exceptions import ClientError

boto3.setup_default_session(profile_name='default', region_name='us-east-1')
lam = boto3.client('lambda')
apigw = boto3.client('apigateway')

Settings

In [ ]:
FUNCTION_NAME = 'dsecountProducts'
SOURCE_FILE = './dsecountProducts/lambda_function.py'
ROLE_ARN = 'arn:aws:iam::199525800117:role/DSmapLambda'   # 기존 약품/기본정보 Lambda 와 같은 역할

API_ID = 'jyipsj28s9'
STAGE = 'dev'
RESOURCE_PATH = 'ecountproducts'
TEMPLATE_FROM = '/dschemical'   # 매핑 템플릿을 복사해 올 기존 리소스

REGION = 'us-east-1'
ACCOUNT_ID = ROLE_ARN.split(':')[4]

1. Lambda 생성 또는 코드 갱신

In [ ]:
buf = io.BytesIO()
with zipfile.ZipFile(buf, 'w', zipfile.ZIP_DEFLATED) as z:
    z.write(SOURCE_FILE, 'lambda_function.py')
zip_bytes = buf.getvalue()

try:
    lam.get_function(FunctionName=FUNCTION_NAME)
    lam.update_function_code(FunctionName=FUNCTION_NAME, ZipFile=zip_bytes)
    print('코드 갱신:', FUNCTION_NAME)
except lam.exceptions.ResourceNotFoundException:
    lam.create_function(
        FunctionName=FUNCTION_NAME, Runtime='python3.13', Role=ROLE_ARN,
        Handler='lambda_function.lambda_handler', Code={'ZipFile': zip_bytes},
        Timeout=10, MemorySize=128,
        Description='이카운트 품목 요약 (s3://dsbaseinfo/ecount/products_latest.json)',
    )
    print('생성:', FUNCTION_NAME)

lam.get_waiter('function_active_v2').wait(FunctionName=FUNCTION_NAME)
lam.get_waiter('function_updated_v2').wait(FunctionName=FUNCTION_NAME)
function_arn = lam.get_function(FunctionName=FUNCTION_NAME)['Configuration']['FunctionArn']
function_arn

2. API Gateway 리소스 `/ecountproducts`

In [ ]:
resources = {r['path']: r for r in apigw.get_resources(restApiId=API_ID, limit=500)['items']}

if f'/{RESOURCE_PATH}' in resources:
    resource_id = resources[f'/{RESOURCE_PATH}']['id']
    print('리소스 있음:', resource_id)
else:
    resource_id = apigw.create_resource(restApiId=API_ID, parentId=resources['/']['id'], pathPart=RESOURCE_PATH)['id']
    print('리소스 생성:', resource_id)

# 기존 /dschemical GET 의 패스스루 매핑 템플릿 복사
src = apigw.get_method(restApiId=API_ID, resourceId=resources[TEMPLATE_FROM]['id'], httpMethod='GET')['methodIntegration']
request_template = src['requestTemplates']['application/json']

3. GET → Lambda (non-proxy) + CORS 헤더

In [ ]:
def put_method_fresh(http_method):
    try:
        apigw.delete_method(restApiId=API_ID, resourceId=resource_id, httpMethod=http_method)
    except apigw.exceptions.NotFoundException:
        pass
    apigw.put_method(restApiId=API_ID, resourceId=resource_id, httpMethod=http_method, authorizationType='NONE')

put_method_fresh('GET')
apigw.put_integration(
    restApiId=API_ID, resourceId=resource_id, httpMethod='GET',
    type='AWS', integrationHttpMethod='POST',
    uri=f'arn:aws:apigateway:{REGION}:lambda:path/2015-03-31/functions/{function_arn}/invocations',
    requestTemplates={'application/json': request_template},
    passthroughBehavior='WHEN_NO_MATCH', contentHandling='CONVERT_TO_TEXT',
)
apigw.put_method_response(
    restApiId=API_ID, resourceId=resource_id, httpMethod='GET', statusCode='200',
    responseParameters={'method.response.header.Access-Control-Allow-Origin': False},
    responseModels={'application/json': 'Empty'},
)
apigw.put_integration_response(
    restApiId=API_ID, resourceId=resource_id, httpMethod='GET', statusCode='200',
    responseParameters={'method.response.header.Access-Control-Allow-Origin': "'*'"},
)
print('GET 설정 완료')

4. OPTIONS (CORS preflight, MOCK)

In [ ]:
put_method_fresh('OPTIONS')
apigw.put_integration(
    restApiId=API_ID, resourceId=resource_id, httpMethod='OPTIONS', type='MOCK',
    requestTemplates={'application/json': '{"statusCode": 200}'}, passthroughBehavior='WHEN_NO_MATCH',
)
cors_headers = {
    'method.response.header.Access-Control-Allow-Headers': "'Content-Type,X-Amz-Date,Authorization,X-Api-Key,X-Amz-Security-Token'",
    'method.response.header.Access-Control-Allow-Methods': "'GET,OPTIONS'",
    'method.response.header.Access-Control-Allow-Origin': "'*'",
}
apigw.put_method_response(
    restApiId=API_ID, resourceId=resource_id, httpMethod='OPTIONS', statusCode='200',
    responseParameters={k: False for k in cors_headers}, responseModels={'application/json': 'Empty'},
)
apigw.put_integration_response(
    restApiId=API_ID, resourceId=resource_id, httpMethod='OPTIONS', statusCode='200',
    responseParameters=cors_headers,
)
print('OPTIONS 설정 완료')

5. API Gateway 가 Lambda 를 호출할 권한

In [ ]:
try:
    lam.add_permission(
        FunctionName=FUNCTION_NAME, StatementId=f'apigw-{API_ID}-get-{RESOURCE_PATH}',
        Action='lambda:InvokeFunction', Principal='apigateway.amazonaws.com',
        SourceArn=f'arn:aws:execute-api:{REGION}:{ACCOUNT_ID}:{API_ID}/*/GET/{RESOURCE_PATH}',
    )
    print('권한 추가')
except lam.exceptions.ResourceConflictException:
    print('권한 이미 있음')

6. `dev` 스테이지 배포 + 확인

In [ ]:
import time
import urllib.error

apigw.create_deployment(restApiId=API_ID, stageName=STAGE, description=f'add GET /{RESOURCE_PATH}')

# 새 경로는 배포 후 반영까지 30초~1분 걸린다 (그 전엔 403 Missing Authentication Token)
url = f'https://{API_ID}.execute-api.{REGION}.amazonaws.com/{STAGE}/{RESOURCE_PATH}'
for attempt in range(12):
    try:
        with urllib.request.urlopen(url) as r:
            res_ = json.load(r)
        break
    except urllib.error.HTTPError as e:
        print(f'{attempt}: HTTP {e.code} - 반영 대기 중...')
        time.sleep(10)
else:
    raise RuntimeError('2분 넘게 응답이 없습니다. 콘솔에서 스테이지 배포를 확인하세요.')

body = json.loads(res_['body'])
print(url)
print('statusCode', res_['statusCode'], '/', body.get('fetched_at'), '/', len(body.get('items', [])), '건' if 'items' in body else body)